# DASC 5131 — Group 2 | A Social Media-Driven Occupational Analysis (v3)

**David Castaneda · Natalie Cluck · Kyoseok Hwang**

---

**Scope**: Computer & Mathematical Occupations (SOC 15-xxxx) + emerging tech roles
(AI / ML / Data / DevOps / Cloud) identified via LinkedIn title keywords.

**Why tech-only?**
- AI-driven title fragmentation is concentrated in computing roles.
- LinkedIn data skews tech-heavy — restricting scope aligns claims with data (Lecture 1: *recognize limitations*).

**DS Lifecycle (Lecture 1-2) mapping**

| Stage | Notebook Phase |
|---|---|
| 1. Question formulation | Phase 0 |
| 2. Data acquisition & cleaning | Phase 1-2 |
| 3. EDA & visualization | Phase 2 |
| 4. Prediction & inference | Phase 3-4 |
| 5. Reports & decisions | Phase 5 |


---
# PHASE 0 — Setup & Research Question  [Stage 1]


### Research Question

> Within U.S. computing occupations, how well does the official SOC/O\*NET taxonomy
> capture the self-reported job titles appearing on LinkedIn — and what does the gap
> reveal about AI-driven occupation change?

**Four sub-objectives (all scoped to SOC 15-xxxx + tech keywords):**

| # | Objective | Method |
|---|---|---|
| O1 | Gap: SOC titles vs LinkedIn self-descriptions | Rule + BERT normalization → unmapped list |
| O2 | Longitudinal: posting volume & AI-title share | `pd.to_datetime` → monthly groupby |
| O3 | AI Impact: skill density from O\*NET | Technology Skills join → ai_density score |
| O4 | Normalization quality + career clusters | Coverage %, K-Means on JobBERT embeddings |


In [ ]:
# [0-1] Install packages (run once, then comment out)
# !pip install pandas numpy openpyxl networkx matplotlib seaborn scikit-learn \
#              sentence-transformers pyarrow tqdm


In [ ]:
# [0-2] Imports
import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
sns.set_style('whitegrid')
print("Imports OK")


In [ ]:
# [0-3] Paths — relative to this notebook's folder
from pathlib import Path

# Notebook lives at .../DASC5131_Group_Project/
# Raw/ and Processed/ are siblings in the same folder
OUTDIR = Path.cwd()
RAW    = OUTDIR / 'Raw'
PROC   = OUTDIR / 'Processed'
(OUTDIR / 'figures').mkdir(exist_ok=True)
(OUTDIR / 'tables').mkdir(exist_ok=True)

assert RAW.exists(),  f"RAW not found: {RAW}"
assert PROC.exists(), f"PROC not found: {PROC}"
print(f"RAW  → {RAW}")
print(f"PROC → {PROC}")


In [ ]:
# [0-4] Tech scope — SOC 15-xxxx + emerging-role keywords
TECH_SOC_PREFIX = ('15-',)

# Positive keywords: roles not yet in SOC but clearly computing
TECH_KW = re.compile(
    r'\b(software|developer|programmer|data scientist|data engineer|data analyst|'
    r'machine learning|ml engineer|ai engineer|prompt engineer|devops|sre|mlops|'
    r'cloud (?:architect|engineer)|cybersecurity|security engineer|'
    r'full[- ]?stack|back[- ]?end|front[- ]?end|qa engineer|'
    r'site reliability|platform engineer)\b',
    flags=re.IGNORECASE,
)

# Exclusions: non-IT "engineer" uses
TECH_KW_EXCL = re.compile(
    r'\b(sales|financial|biomedical|civil|mechanical|chemical|industrial|'
    r'petroleum|environmental|structural) engineer\b',
    flags=re.IGNORECASE,
)


def is_tech_title(title: str) -> bool:
    return bool(TECH_KW.search(title)) and not bool(TECH_KW_EXCL.search(title))


print("Tech scope helpers ready.")


---
# PHASE 1 — Data Loading  [Stage 2a]


### What happens in Phase 1?

Load four datasets, filter to tech at each step — so the entire pipeline works on
a focused universe (~75 O\*NET occupations, ~15-25K LinkedIn rows).

| Dataset | Source | Role |
|---|---|---|
| O\*NET Occupation Data | O\*NET 30.2 | Official SOC codes + titles |
| O\*NET Alternate Titles | O\*NET 30.2 | Rule-based normalization dictionary |
| O\*NET Technology Skills | O\*NET 30.2 | AI/ML skill density (Objective 3) |
| BLS OEWS 2024 | BLS National | Wage & employment per tech occupation |
| LinkedIn postings | Kaggle parquet | Self-reported titles (social media signal) |


In [ ]:
# [1-1] O*NET Occupation Data — keep SOC 15-xxxx only
onet = pd.read_excel(RAW / 'Onet_occupation_data.xlsx')
onet.columns = [c.strip().lower().replace('*', '').replace(' ', '_') for c in onet.columns]

# Identify SOC code column robustly
soc_col = [c for c in onet.columns if 'soc' in c or 'code' in c][0]
onet['soc_6digit'] = onet[soc_col].astype(str).str[:7]

onet_tech = onet[onet['soc_6digit'].str.startswith(TECH_SOC_PREFIX)].copy()
print(f"O*NET all        : {len(onet):,}")
print(f"O*NET tech (15-) : {len(onet_tech):,}")
onet_tech[['soc_6digit', 'title']].head()


In [ ]:
# [1-2] O*NET Alternate Titles — build normalization lookup
alt_candidates = list(RAW.rglob('Alternate Titles.xlsx'))
alt = pd.read_excel(alt_candidates[0])
alt.columns = [c.strip().lower().replace('*', '').replace(' ', '_') for c in alt.columns]
alt['soc_6digit'] = alt[alt.columns[0]].astype(str).str[:7]
alt_tech = alt[alt['soc_6digit'].str.startswith(TECH_SOC_PREFIX)].copy()
print(f"Alternate titles tech: {len(alt_tech):,}  unique: {alt_tech['alternate_title'].nunique():,}")


In [ ]:
# [1-3] O*NET Technology Skills — AI impact analysis
ts_candidates = list(RAW.rglob('Technology Skills.xlsx'))
ts = pd.read_excel(ts_candidates[0])
ts.columns = [c.strip().lower().replace('*', '').replace(' ', '_') for c in ts.columns]
ts['soc_6digit'] = ts[ts.columns[0]].astype(str).str[:7]
ts_tech = ts[ts['soc_6digit'].str.startswith(TECH_SOC_PREFIX)].copy()
print(f"Tech Skills rows     : {len(ts_tech):,}  covering {ts_tech['soc_6digit'].nunique()} occupations")


In [ ]:
# [1-4] BLS OEWS 2024 — wage data for SOC 15-xxxx
xlsx_files = list((RAW / 'oesm24all').rglob('*.xlsx'))
oesm = pd.read_excel(sorted(xlsx_files)[0])
oesm.columns = [c.strip().lower() for c in oesm.columns]
occ_col = next(c for c in oesm.columns if 'occ' in c and 'code' in c)
oesm_tech = oesm[oesm[occ_col].astype(str).str.startswith(TECH_SOC_PREFIX)].copy()
oesm_tech = oesm_tech.rename(columns={occ_col: 'soc_6digit'})
print(f"OEWS tech rows       : {len(oesm_tech):,}")
wage_col = 'a_mean' if 'a_mean' in oesm_tech.columns else oesm_tech.columns[-1]
oesm_tech[['soc_6digit', 'occ_title', wage_col]].head()


In [ ]:
# [1-5] LinkedIn postings — load slim columns, then keyword-filter to tech
parquet_path = PROC / 'linkedin_postings.parquet'
if parquet_path.exists():
    SLIM_COLS = ['title', 'listed_time', 'normalized_salary', 'location']
    try:
        li = pd.read_parquet(parquet_path, columns=SLIM_COLS)
    except Exception:
        li = pd.read_parquet(parquet_path)
        li = li[[c for c in SLIM_COLS if c in li.columns]]
else:
    li = pd.read_csv(PROC / 'postings.csv', usecols=['title', 'listed_time', 'normalized_salary', 'location'])

print(f"LinkedIn raw rows: {len(li):,}")

# Minimal cleaning before filter
li['title_clean'] = (
    li['title'].astype(str)
    .str.lower()
    .str.replace(r'<[^>]+>', '', regex=True)    # strip HTML
    .str.replace(r'[^\x00-\x7f]+', '', regex=True)  # strip non-ASCII / emoji
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Apply tech scope filter
li_tech = li[
    li['title_clean'].str.contains(TECH_KW, na=False)
    & ~li['title_clean'].str.contains(TECH_KW_EXCL, na=False)
].copy()

print(f"LinkedIn tech    : {len(li_tech):,}  ({len(li_tech)/len(li):.1%} of total)")
li_tech[['title_clean', 'listed_time', 'location']].head()


---
# PHASE 2 — Cleaning & EDA  [Stage 2b + Stage 3]


### Exploratory approach (Lecture 2 — "detective work before confirmation")

- Look at distributions and missingness *first*, before any modeling.
- EDA here is **open-ended** — we're discovering patterns, not testing hypotheses yet.


In [ ]:
# [2-1] Parse listed_time Unix-ms → datetime  (Lab4: pd.to_datetime unit='ms')
li_tech['posted_at'] = pd.to_datetime(li_tech['listed_time'], unit='ms', errors='coerce')
li_tech['posted_month'] = li_tech['posted_at'].dt.to_period('M').astype(str)

print("Date range:", li_tech['posted_at'].min(), "→", li_tech['posted_at'].max())
print("Missing dates:", li_tech['posted_at'].isna().sum(),
      f"({li_tech['posted_at'].isna().mean():.1%})")


In [ ]:
# [2-2] Drop duplicates + length filter  (Lab4: drop_duplicates + boolean mask)
before = len(li_tech)
li_tech = li_tech.drop_duplicates(subset=['title_clean', 'location', 'posted_at'])
li_tech = li_tech[li_tech['title_clean'].str.len().between(3, 120)]
print(f"Rows: {before:,} → {len(li_tech):,}  (removed {before - len(li_tech):,})")


In [ ]:
# [2-3] EDA — Top 30 self-reported tech titles
top = li_tech['title_clean'].value_counts().head(30)
fig, ax = plt.subplots(figsize=(8, 9))
top.iloc[::-1].plot(kind='barh', ax=ax, color='#2b6cb0')
ax.set_title('Top 30 Self-Reported Tech Titles (LinkedIn)')
ax.set_xlabel('Posting count')
plt.tight_layout()
plt.savefig(OUTDIR / 'figures' / 'eda_top_titles.png', dpi=120)
plt.show()


In [ ]:
# [2-4] EDA — BLS wage distribution for SOC 15-xxxx
wage_col = 'a_mean' if 'a_mean' in oesm_tech.columns else None
if wage_col:
    plot_df = (
        oesm_tech.dropna(subset=[wage_col])
                 .assign(wage=lambda d: pd.to_numeric(d[wage_col], errors='coerce'))
                 .dropna(subset=['wage'])
                 .sort_values('wage', ascending=False)
                 .head(20)
    )
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(plot_df['occ_title'].str[:50], plot_df['wage'], color='#2f855a')
    ax.invert_yaxis()
    ax.set_title('Annual Mean Wage — Top 20 Computer & Mathematical Occupations (BLS 2024)')
    ax.set_xlabel('Annual Mean Wage (USD)')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'figures' / 'eda_wage.png', dpi=120)
    plt.show()
else:
    print("wage column not found — skip chart")


In [ ]:
# [2-5] EDA — monthly posting volume (longitudinal preview)
monthly = li_tech.dropna(subset=['posted_at']).groupby('posted_month').size()
if len(monthly) > 1:
    fig, ax = plt.subplots(figsize=(11, 4))
    monthly.plot(ax=ax, marker='o', color='#c05621')
    ax.set_title('Tech Posting Volume Over Time (LinkedIn)')
    ax.set_ylabel('Postings')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'figures' / 'eda_timeline.png', dpi=120)
    plt.show()
else:
    print("Not enough time points — timeline skipped")


---
# PHASE 3 — Hybrid Normalization  [Stage 4a]


### Two-layer mapping

1. **Rule layer** — O\*NET Alternate Titles lookup (fast, deterministic).
2. **Semantic layer** — JobBERT cosine similarity for titles the rule misses.

Coverage metric = % of LinkedIn tech rows successfully mapped to a SOC code.


In [ ]:
# [3-1] Abbreviation expansion (common IT shorthand)
ABBREV = {
    'swe': 'software engineer', 'sde': 'software engineer',
    'ds': 'data scientist',     'de': 'data engineer',
    'mle': 'machine learning engineer',
    'sre': 'site reliability engineer',
    'qa':  'quality assurance engineer',
}


def expand_abbrev(s: str) -> str:
    for k, v in ABBREV.items():
        s = re.sub(rf'\b{re.escape(k)}\b', v, s)
    return s


# Build lookup: alt_title_lower → soc_6digit
rule_dict = (
    alt_tech.assign(k=alt_tech['alternate_title'].str.lower().str.strip())
            .drop_duplicates('k')
            .set_index('k')['soc_6digit']
            .to_dict()
)

li_tech['title_exp'] = li_tech['title_clean'].map(expand_abbrev)
li_tech['soc_rule']  = li_tech['title_exp'].map(rule_dict)

cov_rule = li_tech['soc_rule'].notna().mean()
print(f"Rule dictionary  : {len(rule_dict):,} entries")
print(f"Rule coverage    : {cov_rule:.1%}  ({li_tech['soc_rule'].notna().sum():,} rows mapped)")


In [ ]:
# [3-2] JobBERT semantic fallback for rule-miss rows
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('TechWolf/JobBERT-v2')
except Exception:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print("JobBERT unavailable — using MiniLM fallback")

# Canonical set = O*NET tech occupations (SOC 15-*)
canon = onet_tech[['soc_6digit', 'title']].drop_duplicates('soc_6digit').reset_index(drop=True)
canon_emb = model.encode(canon['title'].tolist(), normalize_embeddings=True, show_progress_bar=False)

# Encode only the unmapped rows (sample for speed)
miss = li_tech[li_tech['soc_rule'].isna()].copy()
n_sample = min(3000, len(miss))
miss_s = miss.sample(n_sample, random_state=42) if len(miss) > n_sample else miss

miss_emb   = model.encode(miss_s['title_exp'].tolist(), normalize_embeddings=True, show_progress_bar=False)
sim        = miss_emb @ canon_emb.T
best_idx   = sim.argmax(axis=1)
best_score = sim.max(axis=1)

miss_s = miss_s.copy()
miss_s['soc_bert']    = canon['soc_6digit'].iloc[best_idx].values
miss_s['bert_score']  = best_score
miss_s['bert_title']  = canon['title'].iloc[best_idx].values

COSINE_THRESHOLD = 0.75
accepted = (miss_s['bert_score'] >= COSINE_THRESHOLD).sum()
print(f"Semantic mapped  : {len(miss_s):,} rows  |  accepted (≥{COSINE_THRESHOLD}): {accepted:,}")
miss_s[['title_clean', 'bert_title', 'bert_score']].head(10)


In [ ]:
# [3-3] Merge rule + BERT; report final coverage
li_tech = li_tech.merge(
    miss_s[['title_exp', 'soc_bert', 'bert_score']].drop_duplicates('title_exp'),
    on='title_exp', how='left',
)
li_tech['soc_final'] = li_tech['soc_rule'].fillna(
    li_tech['soc_bert'].where(li_tech.get('bert_score', pd.Series(dtype=float)) >= COSINE_THRESHOLD)
)

cov_final = li_tech['soc_final'].notna().mean()
print(f"Rule layer       : {cov_rule:.1%}")
print(f"Final coverage   : {cov_final:.1%}")
print(f"Still unmapped   : {li_tech['soc_final'].isna().sum():,}")


---
# PHASE 4 — Objectives 1-4  [Stage 4b]


### Objective 1 — Gap Analysis


In [ ]:
# [4-1] Titles that never mapped to a SOC code = the taxonomy gap
gap = (
    li_tech.loc[li_tech['soc_final'].isna(), 'title_clean']
           .value_counts()
           .head(40)
           .rename_axis('title')
           .reset_index(name='postings')
)
gap.to_csv(OUTDIR / 'tables' / 'gap_unmapped_titles.csv', index=False)
print(f"Unmapped unique titles: {li_tech.loc[li_tech['soc_final'].isna(), 'title_clean'].nunique():,}")
gap.head(20)


### Objective 2 — Longitudinal: AI-flavored title share over time


In [ ]:
# [4-2] Monthly share of postings with AI/ML keyword in title
ai_pat = re.compile(r'\b(ai|ml|machine learning|gen ?ai|llm|prompt)\b', re.IGNORECASE)
li_tech['is_ai_title'] = li_tech['title_clean'].str.contains(ai_pat, na=False)

monthly_ai = (
    li_tech.dropna(subset=['posted_at'])
           .groupby('posted_month')['is_ai_title']
           .mean()
           .rename('ai_share')
)
if len(monthly_ai) > 1:
    fig, ax = plt.subplots(figsize=(11, 4))
    monthly_ai.plot(ax=ax, marker='o', color='#805ad5')
    ax.set_title('Share of Tech Postings Mentioning AI/ML in Title (Monthly)')
    ax.set_ylabel('Share')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'figures' / 'obj2_ai_share.png', dpi=120)
    plt.show()
else:
    print("Not enough time points for longitudinal chart")


### Objective 3 — AI Impact: O\*NET Technology Skill Density


In [ ]:
# [4-3] Per-occupation AI/ML skill density from O*NET Technology Skills
ts_tech['is_ai_skill'] = ts_tech['example'].astype(str).str.contains(
    r'artificial intelligence|machine learning|tensorflow|pytorch|scikit|openai|llm|nlp',
    case=False, regex=True, na=False,
)
ai_density = (
    ts_tech.groupby('soc_6digit')
           .agg(total_tools=('example', 'count'), ai_tools=('is_ai_skill', 'sum'))
           .assign(ai_density=lambda d: d['ai_tools'] / d['total_tools'])
           .sort_values('ai_density', ascending=False)
)
# Join O*NET title for readability
ai_density = ai_density.join(onet_tech.set_index('soc_6digit')['title'])
ai_density.to_csv(OUTDIR / 'tables' / 'obj3_ai_skill_density.csv')

# Plot top 15
plot_df = ai_density.dropna(subset=['title']).head(15)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_df['title'].str[:50], plot_df['ai_density'], color='#3182ce')
ax.invert_yaxis()
ax.set_title('AI/ML Skill Density — Top 15 Computer & Mathematical Occupations (O*NET 30.2)')
ax.set_xlabel('Share of listed tech tools that are AI/ML')
plt.tight_layout()
plt.savefig(OUTDIR / 'figures' / 'obj3_ai_density.png', dpi=120)
plt.show()
ai_density.head(10)


### Objective 4 — Career Clusters (K-Means on JobBERT embeddings)


In [ ]:
# [4-4] K-Means clustering on O*NET tech occupation embeddings
from sklearn.cluster import KMeans

k = min(6, len(canon))
km = KMeans(n_clusters=k, n_init=10, random_state=42)
km.fit(canon_emb)
canon = canon.copy()
canon['cluster'] = km.labels_

print("Clusters:")
for cid, grp in canon.groupby('cluster'):
    print(f"  {cid}: {', '.join(grp['title'].head(5).tolist())}")

canon.to_csv(OUTDIR / 'tables' / 'obj4_clusters.csv', index=False)


In [ ]:
# [4-5] Tech occupation network — within SOC 15-xxxx, edge = shared 4-digit group
import networkx as nx

mapped = li_tech.dropna(subset=['soc_final']).copy()
mapped['soc4'] = mapped['soc_final'].str[:5]

# Count postings per soc_final
count = mapped.groupby('soc_final').size().rename('n')

# Build edges: connect two SOC codes that share a 4-digit prefix
pairs = []
for soc4, grp in mapped.groupby('soc4')['soc_final'].unique().items():
    nodes = list(grp)
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            w = count.get(nodes[i], 1) + count.get(nodes[j], 1)
            pairs.append((nodes[i], nodes[j], w))

G = nx.Graph()
G.add_weighted_edges_from(pairs)
title_map = onet_tech.set_index('soc_6digit')['title'].to_dict()
labels = {n: title_map.get(n, n)[:22] for n in G.nodes}
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


In [ ]:
# [4-6] Visualize the tech career network
if G.number_of_nodes() > 0:
    deg = dict(G.degree(weight='weight'))
    clr = [canon.set_index('soc_6digit')['cluster'].get(n, 0) for n in G.nodes]
    fig, ax = plt.subplots(figsize=(12, 9))
    pos = nx.spring_layout(G, seed=42, k=0.6)
    nx.draw_networkx_nodes(G, pos, node_color=clr, cmap='tab10',
                           node_size=[80 + deg[n] for n in G.nodes], alpha=0.85, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
    ax.set_title('Tech Occupation Transition Network — SOC 15-xxxx (color = K-Means cluster)')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'figures' / 'obj4_network.png', dpi=120)
    plt.show()
else:
    print("Not enough mapped rows to build network")


---
# PHASE 5 — Limitations & Conclusion  [Stage 5]


### Limitations

*(Lecture 1: "data is not neutral" — state assumptions honestly)*

- **LinkedIn bias**: high-educated, English-speaking, white-collar skew → findings represent *tech LinkedIn users*, not all U.S. workers.
- **Scope boundary**: conclusions are restricted to SOC 15-xxxx + keyword-matched roles. Generalization to other occupations is out of scope.
- **Normalization threshold**: cosine 0.75 is empirical — no gold-labeled test set. Coverage numbers are approximate.
- **Career transitions inferred**: edges connect SOC codes sharing a 4-digit prefix. Real career history data (bio-job-graph / Twitter) would strengthen Objective 2.
- **Temporal coverage**: depends on LinkedIn dataset; limited longitudinal range.

### Conclusion

Within U.S. computing occupations, the government SOC/O\*NET taxonomy covers the majority of LinkedIn job titles through rule-based matching. However, AI-era roles — Prompt Engineer, ML Engineer, LLM Specialist — fall outside current codes and must be matched semantically. The share of tech postings with AI/ML keywords in their titles shows a rising trend over time. O\*NET Technology Skills already embed AI/ML tools for core computing roles, indicating the *official* taxonomy is catching up, but title diversity on LinkedIn is running ahead. K-Means clusters cleanly separate software, data, security, and ML families, suggesting a SOC 15 sub-split would better reflect current practice.

### Generated outputs

| File | Content |
|---|---|
| `figures/eda_top_titles.png` | Top 30 LinkedIn tech titles |
| `figures/eda_wage.png` | BLS wage by occupation |
| `figures/eda_timeline.png` | Monthly posting volume |
| `figures/obj2_ai_share.png` | AI/ML title share over time |
| `figures/obj3_ai_density.png` | AI skill density per occupation |
| `figures/obj4_network.png` | Tech career network |
| `tables/gap_unmapped_titles.csv` | Titles outside SOC taxonomy |
| `tables/obj3_ai_skill_density.csv` | AI density per SOC code |
| `tables/obj4_clusters.csv` | K-Means cluster assignments |
